# 02. Exploratory Data Analysis

This notebook explores the cleaned used-car dataset produced by the data-inspection notebook. It moves from overall distributions to category comparisons, relationships with selling price, grouped age analysis, and correlation patterns.

The goal is to identify useful patterns and generate questions for the statistical analysis notebook. The visual relationships shown here describe associations, not causal effects.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


def iter_start_paths():
    yield Path.cwd()

    session_name = os.environ.get("JPY_SESSION_NAME")
    if session_name:
        session_path = Path(session_name)
        if not session_path.is_absolute():
            session_path = (Path.cwd() / session_path).resolve()
        yield session_path.parent if session_path.suffix == ".ipynb" else session_path

    ipython_shell = globals().get("get_ipython", lambda: None)()
    if ipython_shell is not None:
        vscode_notebook = ipython_shell.user_ns.get("__vsc_ipynb_file__")
        if vscode_notebook:
            yield Path(vscode_notebook).resolve().parent


def find_project_root() -> Path:
    visited = set()
    for start_path in iter_start_paths():
        for candidate in [start_path, *start_path.parents]:
            if candidate in visited:
                continue
            visited.add(candidate)
            if (candidate / "data").exists() and (candidate / "notebooks").exists():
                return candidate
    raise FileNotFoundError("Could not locate project root containing data/ and notebooks/")


PROJECT_ROOT = find_project_root()
CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "cleaned_car_data.csv"


In [ ]:
if not CLEAN_DATA_PATH.exists():
    raise FileNotFoundError(
        f"{CLEAN_DATA_PATH} was not found. Run notebooks/01_data_inspection.ipynb first to generate it."
    )

df = pd.read_csv(CLEAN_DATA_PATH)
df.head()


In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

## 01. Load and Confirm the Analysis Dataset

The cleaned CSV is loaded from the previous notebook. The shape and column list confirm that the expected derived fields, including `car_age`, are available before exploration begins.

## 02. Numerical Feature Distributions

This section explores the distributions of car age, selling price, and kilometers driven to understand their spread, central tendency, skewness, and potential extreme values.

In [ ]:
numeric_columns = ["car_age", "selling_price", "km_driven"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for column, ax in zip(numeric_columns, axes):
    sns.histplot(data=df, x=column, bins=30, kde=True, ax=ax)
    ax.set_title(f"Distribution of {column.replace('_', ' ').title()}")
    ax.set_xlabel(column.replace("_", " ").title())
    ax.set_ylabel("Number of Cars")

plt.tight_layout()
plt.show()

## 03. Categorical Feature Analysis

This section explores the distribution of categorical variables to understand the composition of the dataset across fuel types, seller types, transmission types, and ownership history.

In [ ]:
categorical_columns = ["fuel", "seller_type", "transmission", "owner"]

for column in categorical_columns:
    print(f"\n{column.upper()}")
    print(df[column].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for column, ax in zip(categorical_columns, axes.flatten()):
    sns.countplot(data=df, x=column, ax=ax)
    ax.set_title(f"Distribution of {column.replace('_', ' ').title()}")
    ax.set_xlabel(column.replace("_", " ").title())
    ax.set_ylabel("Number of Cars")
    
    for container in ax.containers:
        ax.bar_label(container)

plt.tight_layout()
plt.show()

## 04. Categorical Features vs. Selling Price

This section examines how selling prices vary across different categorical groups. Horizontal bar charts compare the median selling price for each category, making differences between groups easy to interpret.

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for column, ax in zip(categorical_columns, axes.flatten()):
    median_prices = (
        df.groupby(column)["selling_price"]
        .median()
        .sort_values(ascending=False)
    )

    sns.barplot(
        x=median_prices.values,
        y=median_prices.index,
        ax=ax
    )

    ax.set_title(
        f"Median Selling Price by {column.replace('_', ' ').title()}"
    )
    ax.set_xlabel("Median Selling Price")
    ax.set_ylabel(column.replace("_", " ").title())

plt.tight_layout()
plt.show()

## 05. Numerical Features vs. Selling Price

This section examines the relationships between selling price and numerical features such as car age and kilometers driven. Scatter plots are used to identify trends, patterns, and potential outliers.

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="car_age",
    y="selling_price",
    alpha=0.8,
    hue = "owner"
)

sns.regplot(
    data=df,
    x="car_age",
    y="selling_price",
    scatter=False
)

plt.title("Car Age vs. Selling Price")
plt.xlabel("Car Age (Years)")
plt.ylabel("Selling Price")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="km_driven",
    y="selling_price",
    alpha=0.8,
    hue = "owner"
)

sns.regplot(
    data=df,
    x="km_driven",
    y="selling_price",
    scatter=False
)

plt.title("Kilometers Driven vs. Selling Price")
plt.xlabel("Kilometers Driven")
plt.ylabel("Selling Price")

plt.tight_layout()
plt.show()

## 06. Selling Price Across Car Age Groups

To make the relationship between vehicle age and selling price easier to interpret, cars are grouped into age ranges. We then compare the median selling price across these groups.

In [ ]:
age_bins = [-1, 3, 6, 10, 15, 20, 30]
age_labels = [
    "0–3 years",
    "4–6 years",
    "7–10 years",
    "11–15 years",
    "16–20 years",
    "21+ years"
]

df["age_group"] = pd.cut(
    df["car_age"],
    bins=age_bins,
    labels=age_labels
)

df["age_group"].value_counts().sort_index()

In [ ]:
age_price = (
    df.groupby("age_group", observed=True)["selling_price"]
    .median()
    .reset_index()
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=age_price,
    x="age_group",
    y="selling_price"
)

plt.title("Median Selling Price by Car Age Group")
plt.xlabel("Car Age")
plt.ylabel("Median Selling Price")

plt.tight_layout()
plt.show()

## 07. Selling Price by Fuel Type & Transmission

This section compares median selling prices across fuel and transmission types. Median prices are used to reduce the influence of extreme vehicle prices, but categories with very small samples should be interpreted cautiously.

In [ ]:
fuel_price = (
    df.groupby("fuel")["selling_price"]
    .median()
    .sort_values(ascending=False)
    .reset_index()
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=fuel_price,
    x="fuel",
    y="selling_price"
)

plt.title("Median Selling Price by Fuel Type")
plt.xlabel("Fuel Type")
plt.ylabel("Median Selling Price")

plt.tight_layout()
plt.show()

In [ ]:
transmission_price = (
    df.groupby("transmission")["selling_price"]
    .median()
    .sort_values(ascending=False)
    .reset_index()
)

plt.figure(figsize=(8, 6))

sns.barplot(
    data=transmission_price,
    x="transmission",
    y="selling_price"
)

plt.title("Median Selling Price by Transmission Type")
plt.xlabel("Transmission")
plt.ylabel("Median Selling Price")

plt.tight_layout()
plt.show()

## 08. Correlation Analysis

This section examines the linear relationships between numerical variables using a correlation matrix. Correlation values range from -1 to 1, where values closer to either -1 or 1 indicate stronger linear relationships.

In [ ]:
numeric_columns = [
    "year",
    "selling_price",
    "km_driven",
    "car_age"
]

correlation_matrix = df[numeric_columns].corr()

plt.figure(figsize=(9, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)

plt.title("Correlation Matrix of Numerical Features")

plt.tight_layout()
plt.show()

## 09. Key EDA Findings

The exploratory analysis revealed several patterns in the used-car dataset:

- The dataset contains a wide range of vehicle ages, selling prices, and kilometers driven.
- Selling price is generally lower for older vehicles, consistent with the negative correlation observed between car age and selling price.
- Vehicle mileage shows a relationship with selling price, although the scatter plot indicates substantial variation between individual vehicles.
- The dataset is heavily concentrated around Diesel and Petrol vehicles, while CNG, LPG, and Electric vehicles have relatively few observations.
- Manual transmission vehicles make up the majority of the dataset.
- Selling prices vary across fuel types, seller types, transmission types, and ownership categories.
- Differences between categories should be interpreted alongside their sample sizes, particularly for categories with very few observations.
- Correlation analysis identifies associations between numerical variables but does not establish causal relationships.

In [ ]:
print("EDA Summary")
print("=" * 40)

print(f"Number of vehicles: {len(df):,}")
print(f"Average selling price: {df['selling_price'].mean():,.0f}")
print(f"Median selling price: {df['selling_price'].median():,.0f}")
print(f"Average car age: {df['car_age'].mean():.1f} years")
print(f"Average kilometers driven: {df['km_driven'].mean():,.0f}")

print("\nStrongest correlations with selling price:")
print(
    correlation_matrix["selling_price"]
    .drop("selling_price")
    .sort_values(key=abs, ascending=False)
)